# Phase 1: MFT Raw Data Parsing

## Overview
This notebook parses the NTFS Master File Table ($MFT) to extract file metadata for timestomping detection analysis.

### Purpose
Extract the following columns from $MFT:
- **Entry Number / FRN**: File Reference Number (MFT record number)
- **$SI-C/M/E/A**: $STANDARD_INFORMATION timestamps (Created, Modified, Entry Modified, Accessed)
- **$FN-C/M/E/A**: $FILE_NAME timestamps
- **File Name**: Name of the file
- **File Path**: Full path to the file
- **Entry Active/Inactive**: Whether the MFT entry is allocated
- **LSN**: LogFile Sequence Number (for cross-referencing with $LogFile)
- **Parent FRN**: Parent directory reference number

### Input
- Raw $MFT file: `data/raw/PE/01-PE/$MFT`

### Output
- Parsed CSV: `data/Phase 1: Raw Data Parsing/01-PE/MFT.csv`

### Library
Uses [dfir_ntfs](https://github.com/bamonskiy-kaban/dfir_ntfs) for NTFS artifact parsing.

---
**Reference**: Oh, Lee, and Hwang (2024) - "Forensic Detection of Timestamp Manipulation for Digital Forensic Investigation"


In [1]:
# [Cell 1] Install and Import Dependencies
# Install dfir_ntfs if not already installed
import subprocess
import sys

def install_package(package):
    """Install a package using pip."""
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

# Install dfir_ntfs from GitHub
try:
    import dfir_ntfs
    print(f"dfir_ntfs version: {dfir_ntfs.__version__}")
except ImportError:
    print("Installing dfir_ntfs...")
    install_package("git+https://github.com/bamonskiy-kaban/dfir_ntfs.git")
    import dfir_ntfs
    print(f"dfir_ntfs installed successfully. Version: {dfir_ntfs.__version__}")


Installing dfir_ntfs...
dfir_ntfs installed successfully. Version: 1.1.19


In [2]:
# [Cell 2] Import Required Libraries
import os
import pandas as pd
from datetime import datetime
from pathlib import Path

# dfir_ntfs modules
from dfir_ntfs import MFT

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

print("Libraries imported successfully.")


Libraries imported successfully.


In [3]:
# [Cell 3] Define Paths and Configuration

# Base project directory
PROJECT_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis")

# Input path - raw MFT file
INPUT_MFT = PROJECT_DIR / "data" / "raw" / "PE" / "01-PE" / "$MFT"

# Output directory
OUTPUT_DIR = PROJECT_DIR / "data" / "Phase 1: Raw Data Parsing" / "01-PE"
OUTPUT_CSV = OUTPUT_DIR / "MFT.csv"

# Create output directory if it does not exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify input file exists
if INPUT_MFT.exists():
    file_size_mb = INPUT_MFT.stat().st_size / (1024 * 1024)
    print(f"Input MFT file: {INPUT_MFT}")
    print(f"File size: {file_size_mb:.2f} MB")
else:
    raise FileNotFoundError(f"MFT file not found: {INPUT_MFT}")

print(f"Output will be saved to: {OUTPUT_CSV}")


Input MFT file: /Users/soni/Github/Digital-Detectives_Thesis/data/raw/PE/01-PE/$MFT
File size: 515.25 MB
Output will be saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 1: Raw Data Parsing/01-PE/MFT.csv


## Timestamp Handling

### NTFS Timestamp Format
- NTFS stores timestamps as 64-bit values representing 100-nanosecond intervals since January 1, 1601 (UTC)
- Python's `datetime` has microsecond precision (6 decimal places)
- To preserve nanosecond precision, we also store the raw FILETIME value

### Timestamp Types
| Attribute | Abbreviation | Description |
|-----------|--------------|-------------|
| $STANDARD_INFORMATION | $SI | User-visible timestamps (can be manipulated) |
| $FILE_NAME | $FN | Updated on file rename/move (harder to manipulate) |

### MACE Timestamps
- **M** - Last Modified Time
- **A** - Last Accessed Time  
- **C** - File Creation Time
- **E** - MFT Entry Modified Time


In [4]:
# [Cell 4] Define Helper Functions for Timestamp Extraction

def format_timestamp(dt_obj):
    """
    Format datetime object to string with microsecond precision.
    Returns None if the timestamp is invalid.
    
    Args:
        dt_obj: datetime object or None
        
    Returns:
        str: ISO format timestamp string or None
    """
    if dt_obj is None:
        return None
    try:
        # Format with full precision (microseconds)
        return dt_obj.strftime("%Y-%m-%d %H:%M:%S.%f")
    except (ValueError, AttributeError):
        return None


def extract_si_timestamps(file_record):
    """
    Extract $STANDARD_INFORMATION timestamps from a file record.
    
    Args:
        file_record: MFT FileRecord object
        
    Returns:
        dict: Dictionary with $SI-C, $SI-M, $SI-E, $SI-A timestamps
    """
    si_timestamps = {
        "SI_C": None,  # Created
        "SI_M": None,  # Modified
        "SI_E": None,  # Entry Modified
        "SI_A": None   # Accessed
    }
    
    try:
        for attribute in file_record.attributes():
            # $STANDARD_INFORMATION has type code 0x10 (16)
            if attribute.type_code == 0x10:
                try:
                    si = attribute.value_decoded()
                    si_timestamps["SI_C"] = format_timestamp(si.get_ctime())
                    si_timestamps["SI_M"] = format_timestamp(si.get_mtime())
                    si_timestamps["SI_E"] = format_timestamp(si.get_etime())
                    si_timestamps["SI_A"] = format_timestamp(si.get_atime())
                    break
                except Exception:
                    pass
    except Exception:
        pass
    
    return si_timestamps


def extract_fn_info(file_record, parser):
    """
    Extract $FILE_NAME attribute information including timestamps.
    Returns the first valid $FN attribute found (prefer long name).
    
    Args:
        file_record: MFT FileRecord object
        parser: MasterFileTableParser instance
        
    Returns:
        dict: Dictionary with file name, parent FRN, and $FN timestamps
    """
    fn_info = {
        "FileName": None,
        "ParentFRN": None,
        "FN_C": None,
        "FN_M": None,
        "FN_E": None,
        "FN_A": None
    }
    
    try:
        for attribute in file_record.attributes():
            # $FILE_NAME has type code 0x30 (48)
            if attribute.type_code == 0x30:
                try:
                    fn = attribute.value_decoded()
                    file_name = fn.get_file_name()
                    
                    # Skip short names (8.3 format) if we already have a long name
                    # Namespace: 0=POSIX, 1=Win32, 2=DOS, 3=Win32+DOS
                    namespace = fn.get_flags()
                    
                    # Prefer Win32 or Win32+DOS names over DOS-only names
                    if fn_info["FileName"] is None or namespace in (1, 3):
                        fn_info["FileName"] = file_name
                        fn_info["ParentFRN"] = fn.get_parent_directory() & 0xFFFFFFFFFFFF  # Lower 48 bits
                        fn_info["FN_C"] = format_timestamp(fn.get_ctime())
                        fn_info["FN_M"] = format_timestamp(fn.get_mtime())
                        fn_info["FN_E"] = format_timestamp(fn.get_etime())
                        fn_info["FN_A"] = format_timestamp(fn.get_atime())
                        
                        # If we found a Win32 name, use it
                        if namespace in (1, 3):
                            break
                except Exception:
                    pass
    except Exception:
        pass
    
    return fn_info


def build_file_path(file_record, parser, path_cache):
    """
    Build the full file path for a given file record.
    Uses caching for performance.
    
    Args:
        file_record: MFT FileRecord object
        parser: MasterFileTableParser instance
        path_cache: Dictionary for caching resolved paths
        
    Returns:
        str: Full file path or None
    """
    try:
        paths = parser.build_full_paths(file_record)
        if paths:
            # Return the first valid path
            return paths[0][0] if isinstance(paths[0], tuple) else paths[0]
    except Exception:
        pass
    return None

print("Helper functions defined successfully.")


Helper functions defined successfully.


In [5]:
# [Cell 5] Parse MFT and Extract Records

def parse_mft(mft_path, progress_interval=50000):
    """
    Parse the MFT file and extract all relevant metadata.
    
    Args:
        mft_path: Path to the $MFT file
        progress_interval: Print progress every N records
        
    Returns:
        list: List of dictionaries containing parsed records
    """
    records = []
    path_cache = {}
    
    print(f"Opening MFT file: {mft_path}")
    
    with open(mft_path, "rb") as mft_file:
        parser = MFT.MasterFileTableParser(mft_file)
        
        record_count = 0
        error_count = 0
        
        print("Parsing MFT records...")
        
        for file_record in parser.file_records():
            try:
                record_count += 1
                
                # Progress indicator
                if record_count % progress_interval == 0:
                    print(f"  Processed {record_count:,} records...")
                
                # Extract basic record info
                entry_number = file_record.get_master_file_table_number()
                is_active = file_record.is_in_use()
                lsn = file_record.get_logfile_sequence_number()
                
                # Extract $SI timestamps
                si_ts = extract_si_timestamps(file_record)
                
                # Extract $FN info and timestamps
                fn_info = extract_fn_info(file_record, parser)
                
                # Build full path
                file_path = build_file_path(file_record, parser, path_cache)
                
                # Combine all info into a record
                record = {
                    "EntryNumber": entry_number,
                    "FileName": fn_info["FileName"],
                    "FilePath": file_path,
                    "IsActive": is_active,
                    "LSN": lsn,
                    "ParentFRN": fn_info["ParentFRN"],
                    "SI_C": si_ts["SI_C"],
                    "SI_M": si_ts["SI_M"],
                    "SI_E": si_ts["SI_E"],
                    "SI_A": si_ts["SI_A"],
                    "FN_C": fn_info["FN_C"],
                    "FN_M": fn_info["FN_M"],
                    "FN_E": fn_info["FN_E"],
                    "FN_A": fn_info["FN_A"]
                }
                
                records.append(record)
                
            except Exception as e:
                error_count += 1
                if error_count <= 5:
                    print(f"  Warning: Error parsing record {record_count}: {str(e)[:100]}")
                elif error_count == 6:
                    print("  (Suppressing further error messages...)")
    
    print(f"\nParsing complete!")
    print(f"  Total records processed: {record_count:,}")
    print(f"  Records with errors: {error_count:,}")
    print(f"  Successfully parsed: {len(records):,}")
    
    return records

# Execute parsing
mft_records = parse_mft(INPUT_MFT)


Opening MFT file: /Users/soni/Github/Digital-Detectives_Thesis/data/raw/PE/01-PE/$MFT
Parsing MFT records...
  Processed 50,000 records...
  Processed 100,000 records...
  Processed 150,000 records...
  Processed 200,000 records...
  Processed 250,000 records...
  Processed 300,000 records...
  Processed 350,000 records...
  Processed 400,000 records...
  Processed 450,000 records...
  Processed 500,000 records...

Parsing complete!
  Total records processed: 519,113
  Records with errors: 0
  Successfully parsed: 519,113


In [6]:
# [Cell 6] Create DataFrame and Organize Columns

# Create DataFrame
df_mft = pd.DataFrame(mft_records)

# Define column order (matching the required output format)
column_order = [
    "EntryNumber",      # Entry Number / FRN
    "FileName",         # File Name
    "FilePath",         # File Path
    "IsActive",         # Entry Active/Inactive
    "LSN",              # LogFile Sequence Number
    "ParentFRN",        # Parent Directory Reference
    "SI_C",             # $SI Created
    "SI_M",             # $SI Modified
    "SI_E",             # $SI Entry Modified
    "SI_A",             # $SI Accessed
    "FN_C",             # $FN Created
    "FN_M",             # $FN Modified
    "FN_E",             # $FN Entry Modified
    "FN_A"              # $FN Accessed
]

# Reorder columns
df_mft = df_mft[column_order]

# Rename columns for clarity
df_mft.columns = [
    "EntryNumber",
    "FileName", 
    "FilePath",
    "IsActive",
    "LSN",
    "ParentFRN",
    "$SI-C",
    "$SI-M",
    "$SI-E",
    "$SI-A",
    "$FN-C",
    "$FN-M",
    "$FN-E",
    "$FN-A"
]

print(f"DataFrame created with {len(df_mft):,} records and {len(df_mft.columns)} columns")
print(f"\nColumn names: {list(df_mft.columns)}")


DataFrame created with 519,113 records and 14 columns

Column names: ['EntryNumber', 'FileName', 'FilePath', 'IsActive', 'LSN', 'ParentFRN', '$SI-C', '$SI-M', '$SI-E', '$SI-A', '$FN-C', '$FN-M', '$FN-E', '$FN-A']


In [7]:
# [Cell 7] Data Quality Summary

print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

# Basic statistics
print(f"\nTotal Records: {len(df_mft):,}")
print(f"Active Entries: {df_mft['IsActive'].sum():,}")
print(f"Inactive (Deleted) Entries: {(~df_mft['IsActive']).sum():,}")

# Missing values
print("\nMissing Values:")
for col in df_mft.columns:
    null_count = df_mft[col].isnull().sum()
    null_pct = (null_count / len(df_mft)) * 100
    print(f"  {col}: {null_count:,} ({null_pct:.2f}%)")

# Sample data
print("\n" + "=" * 60)
print("SAMPLE RECORDS (First 5)")
print("=" * 60)
df_mft.head()


DATA QUALITY SUMMARY

Total Records: 519,113
Active Entries: 517,015
Inactive (Deleted) Entries: 2,098

Missing Values:
  EntryNumber: 0 (0.00%)
  FileName: 4 (0.00%)
  FilePath: 4 (0.00%)
  IsActive: 0 (0.00%)
  LSN: 0 (0.00%)
  ParentFRN: 4 (0.00%)
  $SI-C: 0 (0.00%)
  $SI-M: 0 (0.00%)
  $SI-E: 0 (0.00%)
  $SI-A: 0 (0.00%)
  $FN-C: 4 (0.00%)
  $FN-M: 4 (0.00%)
  $FN-E: 4 (0.00%)
  $FN-A: 4 (0.00%)

SAMPLE RECORDS (First 5)


,EntryNumber,FileName,FilePath,IsActive,LSN,ParentFRN,$SI-C,$SI-M,$SI-E,$SI-A,$FN-C,$FN-M,$FN-E,$FN-A
0,0,$MFT,/$MFT,True,8325153753,5.0,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382
1,1,$MFTMirr,/$MFTMirr,True,33560283,5.0,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382
2,2,$LogFile,/$LogFile,True,33560353,5.0,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382
3,3,$Volume,/$Volume,True,218843902,5.0,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382
4,4,$AttrDef,/$AttrDef,True,33560493,5.0,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382


In [8]:
# [Cell 8] Validate Timestamp Formats

print("=" * 60)
print("TIMESTAMP VALIDATION")
print("=" * 60)

# Check for records with valid $SI timestamps
si_valid = df_mft[df_mft["$SI-C"].notna()].shape[0]
print(f"\nRecords with valid $SI timestamps: {si_valid:,}")

# Check for records with valid $FN timestamps
fn_valid = df_mft[df_mft["$FN-C"].notna()].shape[0]
print(f"Records with valid $FN timestamps: {fn_valid:,}")

# Check for records where $SI-C != $FN-C (potential timestomping indicator)
both_valid = df_mft[(df_mft["$SI-C"].notna()) & (df_mft["$FN-C"].notna())]
si_fn_mismatch = both_valid[both_valid["$SI-C"] != both_valid["$FN-C"]]
print(f"\nRecords where $SI-C != $FN-C: {len(si_fn_mismatch):,}")

# Sample of timestamp data
print("\n" + "=" * 60)
print("SAMPLE TIMESTAMPS (First 5 records with complete data)")
print("=" * 60)
sample_cols = ["EntryNumber", "FileName", "$SI-C", "$FN-C"]
df_mft[df_mft["$SI-C"].notna()][sample_cols].head()


TIMESTAMP VALIDATION

Records with valid $SI timestamps: 519,113
Records with valid $FN timestamps: 519,109

Records where $SI-C != $FN-C: 102,034

SAMPLE TIMESTAMPS (First 5 records with complete data)


,EntryNumber,FileName,$SI-C,$FN-C
0,0,$MFT,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382
1,1,$MFTMirr,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382
2,2,$LogFile,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382
3,3,$Volume,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382
4,4,$AttrDef,2022-12-16 08:09:43.729382,2022-12-16 08:09:43.729382


In [9]:
# [Cell 9] Save to CSV

# Save DataFrame to CSV
df_mft.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

# Verify the saved file
saved_size = OUTPUT_CSV.stat().st_size / (1024 * 1024)
print(f"MFT data saved to: {OUTPUT_CSV}")
print(f"Output file size: {saved_size:.2f} MB")

# Verify by reading back
df_verify = pd.read_csv(OUTPUT_CSV)
print(f"Verification: {len(df_verify):,} records saved successfully")


MFT data saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 1: Raw Data Parsing/01-PE/MFT.csv
Output file size: 223.24 MB
Verification: 519,113 records saved successfully


In [10]:
# [Cell 10] Final Summary and Statistics

print("=" * 60)
print("PHASE 1 - MFT PARSING COMPLETE")
print("=" * 60)

print(f"""
Input:  {INPUT_MFT}
Output: {OUTPUT_CSV}

Statistics:
-----------
Total MFT Entries:     {len(df_mft):,}
Active Entries:        {df_mft['IsActive'].sum():,}
Deleted Entries:       {(~df_mft['IsActive']).sum():,}
With File Names:       {df_mft['FileName'].notna().sum():,}
With File Paths:       {df_mft['FilePath'].notna().sum():,}
With $SI Timestamps:   {df_mft['$SI-C'].notna().sum():,}
With $FN Timestamps:   {df_mft['$FN-C'].notna().sum():,}

Columns Extracted:
------------------
{chr(10).join(f'  - {col}' for col in df_mft.columns)}

Ready for Phase 2: Data Preprocessing and Event Grouping
""")


PHASE 1 - MFT PARSING COMPLETE

Input:  /Users/soni/Github/Digital-Detectives_Thesis/data/raw/PE/01-PE/$MFT
Output: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 1: Raw Data Parsing/01-PE/MFT.csv

Statistics:
-----------
Total MFT Entries:     519,113
Active Entries:        517,015
Deleted Entries:       2,098
With File Names:       519,109
With File Paths:       519,109
With $SI Timestamps:   519,113
With $FN Timestamps:   519,109

Columns Extracted:
------------------
  - EntryNumber
  - FileName
  - FilePath
  - IsActive
  - LSN
  - ParentFRN
  - $SI-C
  - $SI-M
  - $SI-E
  - $SI-A
  - $FN-C
  - $FN-M
  - $FN-E
  - $FN-A

Ready for Phase 2: Data Preprocessing and Event Grouping

